# Assignment 2 — Normalized TF and IDF

Same dataset as Assignment 1 (23 Indian languages). We implement:

1. **TF normalization** — (a) by total words in the sentence, (b) by the most frequent word in the sentence
2. **IDF normalization** — by the document frequency of the most frequent word across sentences
3. **Logistic regression on all 6 combinations** (3 TF choices x 2 IDF choices) and compare

Two differences from Assignment 1, both required for the comparison to mean anything:

* **No L2 row normalization** — it would cancel the TF normalizations exactly, making all three TF schemes identical.
* **Word unigrams only** — the TF definitions are stated in terms of words.
* **No vocabulary cap** — see section 3.1.

## 0. Settings

In [1]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from scipy import sparse

MAX_FEATURES = None            # None = keep every word (see section 3.1)

EPOCHS        = 30
LEARNING_RATE = 1.0
BATCH_SIZE    = 128

SEED = 42
DATA_FILE = os.path.join("data", "langid_dataset.csv")

## 1. Dataset (from Assignment 1)

In [2]:
LANGUAGES = {
    "as": "Assamese", "bd": "Bodo",      "bn": "Bengali",   "dg": "Dogri",
    "gom": "Konkani", "gu": "Gujarati",  "hi": "Hindi",     "kha": "Khasi",
    "kn": "Kannada",  "ks": "Kashmiri",  "mai": "Maithili", "ml": "Malayalam",
    "mni": "Manipuri","mr": "Marathi",   "ne": "Nepali",    "or": "Odia",
    "pa": "Punjabi",  "sa": "Sanskrit",  "sat": "Santali",  "sd": "Sindhi",
    "ta": "Tamil",    "te": "Telugu",    "ur": "Urdu",
}

LABELS = sorted(LANGUAGES)                             # label 0 = "as", label 1 = "bd", ...
label_to_number = {code: i for i, code in enumerate(LABELS)}

dataset = pd.read_csv(DATA_FILE)                       # same file as Assignment 1
print(f"{len(dataset)} sentences, {dataset.groupby('label').size().iloc[0]} per language")
dataset.head()

23000 sentences, 1000 per language


,label,text
0,as,ঘটনাৰ বিৱৰণি মতে বোকাকাতৰ কৈলাখাতত নিয়মিতভাৱে ...
1,as,বোকাখাতৰ কৈলাখাতত নিয়মিতভাৱে বহা এই জুৱাৰ থলীত...
2,as,ৰাণু মণ্ডলৰ সুৱদী কন্ঠত মুগ্ধ হৈ ভাইজান ছলমানে...
3,as,"বহুতে ইয়াকো কৈছে যে, ছলমানৰ আহিবলগীয়া ছবিৰ বাব..."
4,as,কিন্তু আঁচল কথাতো হল ছচিয়েল মিডিয়াত প্ৰচাৰিত এ...


## 2. Stratified 80 / 10 / 10 split

In [3]:
def stratified_split(table):
    rng = np.random.default_rng(SEED)
    train, validation, test = [], [], []

    for label, group in table.groupby("label"):         # split inside each language
        group = group.iloc[rng.permutation(len(group))]
        n_train = int(0.8 * len(group))
        n_validation = int(0.1 * len(group))

        train.append(group.iloc[:n_train])
        validation.append(group.iloc[n_train:n_train + n_validation])
        test.append(group.iloc[n_train + n_validation:])

    return [pd.concat(p).sample(frac=1, random_state=SEED) for p in (train, validation, test)]


train_set, validation_set, test_set = stratified_split(dataset)

train_labels      = train_set["label"].map(label_to_number).to_numpy()
validation_labels = validation_set["label"].map(label_to_number).to_numpy()
test_labels       = test_set["label"].map(label_to_number).to_numpy()

for name, part in [("train", train_set), ("validation", validation_set), ("test", test_set)]:
    print(f"{name:<11} {len(part):>6} sentences")

train        18400 sentences
validation    2300 sentences
test          2300 sentences


## 3. Tokenization — a word is a whitespace-separated token

In [4]:
def tokenize(sentence):
    return sentence.lower().split()                     # a "word" = whitespace-separated token


train_documents      = [tokenize(t) for t in train_set["text"]]
validation_documents = [tokenize(t) for t in validation_set["text"]]
test_documents       = [tokenize(t) for t in test_set["text"]]

tokenize("The cat sat on the mat the cat")

['the', 'cat', 'sat', 'on', 'the', 'mat', 'the', 'cat']

### 3.1 Vocabulary and document frequency

`df.max()` = document frequency of the most frequent word across sentences (used by task 2).

No vocabulary cap (unlike Assignment 1): ~75% of words have `df == 1`, a cap would keep an
arbitrary subset of them, and every word it would drop is unique to a single language — the
most discriminative features there are, and the ones task 2 up-weights.

In [5]:
# Document frequency: in how many TRAINING sentences does each word appear?
document_frequency = Counter()
for document in train_documents:
    document_frequency.update(set(document))

top = document_frequency.most_common(MAX_FEATURES)   # None -> every word, sorted by df
vocabulary = {word: column for column, (word, _) in enumerate(top)}

df_counts = np.array([count for _, count in top], dtype=float)   # df of each vocabulary word
n_documents = len(train_documents)

print(f"vocabulary       {len(vocabulary):,} words")
print(f"of which df == 1 {int((df_counts == 1).sum()):,} words")
print(f"n_documents (N)  {n_documents:,}")
print(f"most frequent word across sentences: '{top[0][0]}'  df = {int(df_counts.max())}")

vocabulary       108,126 words
of which df == 1 81,139 words
n_documents (N)  18,400
most frequent word across sentences: 'के'  df = 634


## 4. Raw counts + per-sentence normalizers

Built once; the six combinations rescale this same matrix.

In [6]:
def count_matrix(documents):
    """Raw word counts + the two per-sentence normalizers we will need."""
    rows, columns, values = [], [], []
    total_words, max_word_frequency = [], []

    for row, document in enumerate(documents):
        counts = Counter(document)
        total_words.append(max(len(document), 1))                    # |d|
        max_word_frequency.append(max(counts.values()) if counts else 1)   # max_w f(w, d)

        for word, count in counts.items():
            column = vocabulary.get(word)
            if column is not None:                                   # unseen words are ignored
                rows.append(row)
                columns.append(column)
                values.append(float(count))                          # raw TF

    matrix = sparse.csr_matrix((values, (rows, columns)),
                               shape=(len(documents), len(vocabulary)))
    return matrix, np.array(total_words, float), np.array(max_word_frequency, float)


train_counts,      train_totals,      train_maxima      = count_matrix(train_documents)
validation_counts, validation_totals, validation_maxima = count_matrix(validation_documents)
test_counts,       test_totals,       test_maxima       = count_matrix(test_documents)

print(train_counts.shape, f"{train_counts.nnz:,} non-zeros")

(18400, 108126) 249,532 non-zeros


## 5. Task 1 — three TF schemes

* `raw` — $f(w,d)$ (unnormalized)
* `length` — 1(a): $f(w,d) / \lvert d \rvert$
* `max` — 1(b): $f(w,d) / \max_{w'} f(w',d)$

In [7]:
def term_frequency(counts, totals, maxima, scheme):
    if scheme == "raw":                                  # unnormalized (Assignment 1)
        return counts
    divisor = totals if scheme == "length" else maxima    # 1(a) or 1(b)
    return sparse.diags(1 / divisor) @ counts             # divide every row by one number


TF_SCHEMES = ["raw", "length", "max"]

Example on `"the cat sat on the mat the cat"`:

In [8]:
demo = tokenize("the cat sat on the mat the cat")
demo_counts = Counter(demo)

pd.DataFrame({
    "raw f(w,d)":      dict(demo_counts),
    "1(a)  / |d|":     {w: c / len(demo) for w, c in demo_counts.items()},
    "1(b)  / max f":   {w: c / max(demo_counts.values()) for w, c in demo_counts.items()},
}).round(3)

,"raw f(w,d)",1(a) / |d|,1(b) / max f
the,3,0.375,1.000
cat,2,0.250,0.667
sat,1,0.125,0.333
on,1,0.125,0.333
mat,1,0.125,0.333


## 6. Task 2 — two IDF schemes

* plain (unnormalized): $\log\frac{1+N}{1+\mathrm{df}(w)} + 1$
* normalized: $\log\!\left(1 + \frac{\max_{w'}\mathrm{df}(w')}{\mathrm{df}(w)}\right)$

The normalized version spreads the weights over a much wider range.

In [9]:
idf_plain = np.log((1 + n_documents) / (1 + df_counts)) + 1     # unnormalized (Assignment 1)
idf_norm  = np.log(1 + df_counts.max() / df_counts)             # 2. normalized by df of the
                                                                #    most frequent word
IDF_SCHEMES = {"plain": idf_plain, "norm": idf_norm}

pd.DataFrame({
    "word":       [w for w, _ in top[:5]] + [w for w, _ in top[-3:]],
    "df":         list(df_counts[:5])     + list(df_counts[-3:]),
    "idf plain":  list(idf_plain[:5])     + list(idf_plain[-3:]),
    "idf norm":   list(idf_norm[:5])      + list(idf_norm[-3:]),
}).round(3)

,word,df,idf plain,idf norm
0,के,634.0,4.367,0.693
1,में,492.0,4.620,0.828
2,ꯑꯁꯤ,490.0,4.624,0.830
3,the,467.0,4.672,0.858
4,جي,432.0,4.749,0.903
5,اصفهاني,1.0,10.127,6.454
6,چوڌاري,1.0,10.127,6.454
7,اسپارڪو,1.0,10.127,6.454


## 7. TF-IDF matrix: $X[d,w] = \mathrm{tf}(w,d)\times \mathrm{idf}(w)$

In [10]:
def build(scheme_tf, scheme_idf):
    idf = IDF_SCHEMES[scheme_idf]
    parts = [(train_counts, train_totals, train_maxima),
             (validation_counts, validation_totals, validation_maxima),
             (test_counts, test_totals, test_maxima)]
    return [term_frequency(c, t, m, scheme_tf).multiply(idf).tocsr() for c, t, m in parts]


X_train, X_validation, X_test = build("length", "norm")
print(X_train.shape, f"mean non-zero value {X_train.data.mean():.4f}")

(18400, 108126) mean non-zero value 0.3487


## 8. Logistic regression (softmax) — same model as Assignment 1

In [11]:
def softmax(scores):
    scores = scores - scores.max(axis=1, keepdims=True)      # prevents exp() overflow
    exponentials = np.exp(scores)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


class LogisticRegression:

    def fit(self, X, y, verbose=False):
        n_samples, n_features = X.shape
        n_classes = y.max() + 1

        self.W = np.zeros((n_features, n_classes))
        self.b = np.zeros(n_classes)
        rng = np.random.default_rng(SEED)

        for epoch in range(1, EPOCHS + 1):
            for batch in np.array_split(rng.permutation(n_samples), n_samples // BATCH_SIZE):
                X_batch = X[batch]

                error = softmax(X_batch @ self.W + self.b)
                error[np.arange(len(batch)), y[batch]] -= 1   # error = P - Y  (Y is one-hot)
                error /= len(batch)

                self.W -= LEARNING_RATE * (X_batch.T @ error)
                self.b -= LEARNING_RATE * error.sum(axis=0)

            if verbose and epoch % 10 == 0:
                probabilities = softmax(X @ self.W + self.b)
                loss = -np.log(probabilities[np.arange(n_samples), y] + 1e-12).mean()
                print(f"  epoch {epoch:>3}   loss {loss:.4f}")

        return self

    def predict(self, X):
        return np.asarray(X @ self.W + self.b).argmax(axis=1)

## 9. Macro-F1 — same as Assignment 1

In [12]:
def confusion_matrix(true_labels, predicted_labels):
    counts = np.zeros((len(LABELS), len(LABELS)), dtype=int)
    for true, predicted in zip(true_labels, predicted_labels):
        counts[true][predicted] += 1                    # rows = true, columns = predicted
    return counts


def divide(a, b):
    return a / b if b else 0.0                          # 0/0 counts as 0, by convention


def precision_recall_f1(true_labels, predicted_labels):
    counts = confusion_matrix(true_labels, predicted_labels)
    precisions, recalls, f1_scores = [], [], []

    for k in range(len(LABELS)):
        true_positives  = counts[k][k]
        false_positives = counts[:, k].sum() - true_positives
        false_negatives = counts[k, :].sum() - true_positives

        precision = divide(true_positives, true_positives + false_positives)
        recall    = divide(true_positives, true_positives + false_negatives)

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(divide(2 * precision * recall, precision + recall))

    return np.array(precisions), np.array(recalls), np.array(f1_scores), counts


def macro_f1(true_labels, predicted_labels):
    return precision_recall_f1(true_labels, predicted_labels)[2].mean()

## 10. Task 3 — all 6 combinations

Same data, model and hyper-parameters throughout; only TF/IDF changes. Takes a few minutes.

In [13]:
TF_NAMES  = {"raw": "TF unnormalized", "length": "TF / total words", "max": "TF / max word freq"}
IDF_NAMES = {"plain": "IDF unnormalized", "norm": "IDF normalized"}

results = []
models  = {}

for scheme_tf in TF_SCHEMES:
    for scheme_idf in IDF_SCHEMES:
        X_train, X_validation, X_test = build(scheme_tf, scheme_idf)

        classifier = LogisticRegression().fit(X_train, train_labels)
        models[(scheme_tf, scheme_idf)] = (classifier, X_test)

        validation_predictions = classifier.predict(X_validation)
        test_predictions       = classifier.predict(X_test)

        results.append({
            "TF":  TF_NAMES[scheme_tf],
            "IDF": IDF_NAMES[scheme_idf],
            "val accuracy":  (validation_predictions == validation_labels).mean(),
            "val macro-F1":  macro_f1(validation_labels, validation_predictions),
            "test accuracy": (test_predictions == test_labels).mean(),
            "test macro-F1": macro_f1(test_labels, test_predictions),
        })
        print(f"{TF_NAMES[scheme_tf]:<20} {IDF_NAMES[scheme_idf]:<18} "
              f"test macro-F1 {results[-1]['test macro-F1']:.4f}")

results_table = pd.DataFrame(results).round(4)
results_table

TF unnormalized      IDF unnormalized   test macro-F1 0.9126


TF unnormalized      IDF normalized     test macro-F1 0.9219


TF / total words     IDF unnormalized   test macro-F1 0.9132


TF / total words     IDF normalized     test macro-F1 0.9232


TF / max word freq   IDF unnormalized   test macro-F1 0.9168


TF / max word freq   IDF normalized     test macro-F1 0.9316


,TF,IDF,val accuracy,val macro-F1,test accuracy,test macro-F1
0,TF unnormalized,IDF unnormalized,0.9343,0.9373,0.9083,0.9126
1,TF unnormalized,IDF normalized,0.9361,0.9396,0.9165,0.9219
2,TF / total words,IDF unnormalized,0.9265,0.9296,0.9104,0.9132
3,TF / total words,IDF normalized,0.9352,0.9390,0.9178,0.9232
4,TF / max word freq,IDF unnormalized,0.9370,0.9395,0.9135,0.9168
5,TF / max word freq,IDF normalized,0.9435,0.9453,0.9291,0.9316


### Ranked by test macro-F1

In [14]:
results_table.sort_values("test macro-F1", ascending=False).reset_index(drop=True)

,TF,IDF,val accuracy,val macro-F1,test accuracy,test macro-F1
0,TF / max word freq,IDF normalized,0.9435,0.9453,0.9291,0.9316
1,TF / total words,IDF normalized,0.9352,0.9390,0.9178,0.9232
2,TF unnormalized,IDF normalized,0.9361,0.9396,0.9165,0.9219
3,TF / max word freq,IDF unnormalized,0.9370,0.9395,0.9135,0.9168
4,TF / total words,IDF unnormalized,0.9265,0.9296,0.9104,0.9132
5,TF unnormalized,IDF unnormalized,0.9343,0.9373,0.9083,0.9126


### 3 x 2 grid (test macro-F1)

In [15]:
results_table.pivot(index="TF", columns="IDF", values="test macro-F1") \
             .loc[list(TF_NAMES.values()), list(IDF_NAMES.values())]

IDF,IDF unnormalized,IDF normalized
TF,,
TF unnormalized,0.9126,0.9219
TF / total words,0.9132,0.9232
TF / max word freq,0.9168,0.9316


### Effect of each choice

In [16]:
pivot = results_table.pivot(index="TF", columns="IDF", values="test macro-F1")

baseline = pivot.loc["TF unnormalized", "IDF unnormalized"]        # the Assignment-1 setting
best_row = results_table.loc[results_table["test macro-F1"].idxmax()]

print(f"Assignment-1 baseline (both unnormalized) : {baseline:.4f}")
print(f"Best combination ({best_row['TF']} + {best_row['IDF']}) : {best_row['test macro-F1']:.4f}")
print(f"Improvement                               : {best_row['test macro-F1'] - baseline:+.4f}\n")

print("Average effect of normalizing IDF (over the 3 TF schemes):"
      f" {(pivot['IDF normalized'] - pivot['IDF unnormalized']).mean():+.4f}")
print("Average effect of each TF scheme (over the 2 IDF schemes):")
for name in TF_NAMES.values():
    print(f"  {name:<20} {pivot.loc[name].mean():.4f}")

Assignment-1 baseline (both unnormalized) : 0.9126
Best combination (TF / max word freq + IDF normalized) : 0.9316
Improvement                               : +0.0190

Average effect of normalizing IDF (over the 3 TF schemes): +0.0114
Average effect of each TF scheme (over the 2 IDF schemes):
  TF unnormalized      0.9173
  TF / total words     0.9182
  TF / max word freq   0.9242


### Per-language breakdown, best combination

In [17]:
best_key = max(models, key=lambda k: macro_f1(test_labels, models[k][0].predict(models[k][1])))
classifier, X_test = models[best_key]
test_predictions = classifier.predict(X_test)

precisions, recalls, f1_scores, counts = precision_recall_f1(test_labels, test_predictions)
print(f"Best combination: {TF_NAMES[best_key[0]]} + {IDF_NAMES[best_key[1]]}   "
      f"macro-F1 = {f1_scores.mean():.4f}\n")

pd.DataFrame({
    "language":  [LANGUAGES[c] for c in LABELS],
    "precision": precisions.round(3),
    "recall":    recalls.round(3),
    "f1":        f1_scores.round(3),
    "support":   counts.sum(axis=1),
}, index=LABELS).sort_values("f1")

Best combination: TF / max word freq + IDF normalized   macro-F1 = 0.9316



,language,precision,recall,f1,support
gom,Konkani,0.696,0.71,0.703,100
mr,Marathi,0.739,0.68,0.708,100
sa,Sanskrit,0.603,0.94,0.734,100
hi,Hindi,0.828,0.96,0.889,100
bd,Bodo,0.988,0.84,0.908,100
dg,Dogri,0.938,0.91,0.924,100
ml,Malayalam,1.000,0.89,0.942,100
mai,Maithili,0.959,0.93,0.944,100
kn,Kannada,1.000,0.90,0.947,100
ta,Tamil,0.989,0.93,0.959,100


## 11. Conclusion

* **Normalizing IDF helps** under all three TF schemes (+0.0093, +0.0100, +0.0148) — the most reliable result here, since three independent comparisons agree on the sign. Its weight range is 9.3x versus 2.3x for the plain IDF, so rare words are separated far more sharply.
* **Normalizing TF helps**, and 1(b) (÷ max word frequency) beats 1(a) (÷ sentence length). 1(a) is only ~0.001 ahead of raw counts, which is not a real gap.
* **Best combination: 1(b) + normalized IDF = 0.9316**, versus 0.9126 for the fully unnormalized Assignment-1 setting (**+0.0190**). Fully unnormalized is the worst cell.
* **Noise floor:** with 2300 test sentences the standard error is ~0.006, so sub-half-point differences are not meaningful. The IDF conclusion survives this; the raw-vs-1(a) ordering does not.
* Absolute scores sit below Assignment 1 (0.9592) because features are word unigrams only and L2 row normalization was dropped — both required for this comparison to mean anything. The relative comparison is unaffected.
* Capping the vocabulary at 50k (as Assignment 1 does) **reversed** the TF ordering (1(a) 0.9149 > 1(b) 0.9038) by discarding 58,126 language-unique, high-IDF words. An inherited preprocessing default can flip the conclusion.